# E-Ticaret Oturum Analizi
### Ziyaretçi Davranışlarından Satın Alma Tahmini

**Hazırlayan:** Damla Karaman

## Problem ve amaç
Bir e-ticaret sitesindeki ziyaretçinin gezinme davranışlarını kullanarak,
oturumun satın almayla sonuçlanıp sonuçlanmadığını tahmin etmek.

Bu çalışma, satın almayla ilişkili davranışları inceleyen ve geçmiş
oturum özetleri üzerinde çalışan bir sınıflandırma prototipidir.

## Hedef değişken
Revenue:
- False: Satın alma gerçekleşmedi.
- True: Satın alma gerçekleşti.

## Yöntem
Önce çoğunluk sınıfını tahmin eden bir referans yöntem
(DummyClassifier), ardından bir karar ağacı (DecisionTreeClassifier)
kullanılacaktır.

## Veri kaynağı
Online Shoppers Purchasing Intention Dataset — UCI.
Kaggle: https://www.kaggle.com/datasets/imakash3011/online-shoppers-purchasing-intention-dataset

## Proje kararları ve rapor notları
- Revenue hedef değişkendir; model girdilerine dahil edilmeyecektir.
- PageValues, hesaplanmasında işlem değerleriyle ilişkili bilgi
  kullanıldığı için ilk modelden çıkarılacaktır. Bu karar, doğrulanmış
  veri sızıntısı bulgusu olarak sunulmayacaktır.
- Sınıf dağılımı incelenecek; accuracy yanında precision, recall ve F1
  raporlanacaktır.
- Ön işleme yalnızca eğitim verisinden öğrenilecektir.
- Model ayarları eğitim verisi içinde seçilecek; test verisi son
  değerlendirme için ayrılacaktır.
- Tamamlanmış oturum özetleriyle çalışıldığı için gerçek zamanlı
  tahmin performansı iddia edilmeyecektir.
- Model sonuçları, indirim veya kampanyanın nedensel etkisini göstermez.
- Bulgular ve başarı değerleri kod çıktıları alındıktan sonra yazılacaktır.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path
import hashlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Drive'daki klasör adını ekran görüntüsüne göre kullanıyoruz.
PROJE_KLASORU = Path("/content/drive/MyDrive/BasitlestirilmisPy")
VERI_DOSYASI = PROJE_KLASORU / "online_shoppers_intention.csv"
CIKTI_KLASORU = PROJE_KLASORU / "ciktilar"

if not VERI_DOSYASI.is_file():
    raise FileNotFoundError(
        f"CSV bulunamadı: {VERI_DOSYASI}\n"
        "Drive bağlantısını, klasör adını ve CSV dosya adını kontrol et."
    )

CIKTI_KLASORU.mkdir(parents=True, exist_ok=True)

print("CSV bulundu:", VERI_DOSYASI.name)
print("Çıktı klasörü hazır:", CIKTI_KLASORU)
print("Pandas sürümü:", pd.__version__)

CSV bulundu: online_shoppers_intention.csv
Çıktı klasörü hazır: /content/drive/MyDrive/BasitlestirilmisPy/ciktilar
Pandas sürümü: 2.2.3


In [6]:
# Ham veriyi koruyor, inceleme için ayrı bir kopya oluşturuyoruz.
df_ham = pd.read_csv(VERI_DOSYASI)
df = df_ham.copy()

# Aynı veri dosyasını kullandığımızı kontrol edebilmek için parmak izi.
veri_sha256 = hashlib.sha256(VERI_DOSYASI.read_bytes()).hexdigest()

print(f"Satır sayısı: {df.shape[0]:,}")
print(f"Sütun sayısı: {df.shape[1]}")
print("Veri dosyası SHA-256:", veri_sha256)

display(df.head())

# Veri kaynağı bilgilerini Drive'a kaydet.
kaynak_notu = (
    "Veri seti: Online Shoppers Purchasing Intention Dataset\n"
    "Asıl kaynak: UCI Machine Learning Repository\n"
    "Kaggle: https://www.kaggle.com/datasets/imakash3011/"
    "online-shoppers-purchasing-intention-dataset\n"
    f"Dosya: {VERI_DOSYASI.name}\n"
    f"Boyut: {df.shape[0]} satır, {df.shape[1]} sütun\n"
    f"SHA-256: {veri_sha256}\n"
)

(CIKTI_KLASORU / "veri_kaynagi.txt").write_text(
    kaynak_notu, encoding="utf-8"
)

Satır sayısı: 12,330
Sütun sayısı: 18
Veri dosyası SHA-256: b3055ee355f59134d851d32641183cb4a8b45def7124d2f50442a042f358e0d9


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


338

In [7]:
if "Revenue" not in df.columns:
    raise ValueError("Revenue sütunu bulunamadı.")

if df["Revenue"].isna().any():
    raise ValueError("Hedef sütununda eksik değer var; incelemeliyiz.")

print("Revenue veri tipi:", df["Revenue"].dtype)
print("Revenue değerleri:", df["Revenue"].unique())

hedef_ozeti = df["Revenue"].value_counts(dropna=False).to_frame(
    name="oturum_sayisi"
)
hedef_ozeti["oran_yuzde"] = (
    hedef_ozeti["oturum_sayisi"] / len(df) * 100
).round(2)

display(hedef_ozeti)

hedef_ozeti.to_csv(
    CIKTI_KLASORU / "hedef_dagilimi.csv",
    encoding="utf-8-sig",
    index_label="Revenue"
)

Revenue veri tipi: bool
Revenue değerleri: [False  True]


,oturum_sayisi,oran_yuzde
Revenue,,
False,10422,84.53
True,1908,15.47


In [8]:
print("SÜTUNLAR VE VERİ TİPLERİ")
df.info()

kalite_ozeti = pd.DataFrame({
    "veri_tipi": df.dtypes.astype(str),
    "eksik_deger": df.isna().sum(),
    "benzersiz_deger": df.nunique(dropna=False)
})

display(kalite_ozeti)

# Sayısal sütunlarda negatif ve sonsuz değer kontrolü.
sayisal_veri = df.select_dtypes(include="number")

sayisal_kontrol = pd.DataFrame({
    "minimum": sayisal_veri.min(),
    "maksimum": sayisal_veri.max(),
    "negatif_deger": (sayisal_veri < 0).sum(),
    "sonsuz_deger": np.isinf(sayisal_veri).sum()
})

display(sayisal_kontrol)

print("Toplam eksik hücre:", int(df.isna().sum().sum()))
print("Tam satır tekrar sayısı:", int(df.duplicated().sum()))

# Aynı girdilere sahip kayıtların hedefleri farklı mı?
girdi_sutunlari = df.columns.drop("Revenue").tolist()

celiskili_gruplar = (
    df.groupby(girdi_sutunlari, dropna=False)["Revenue"]
    .nunique()
    .gt(1)
    .sum()
)

print("Aynı girdilere rağmen farklı hedef içeren grup:", int(celiskili_gruplar))

kalite_ozeti.to_csv(
    CIKTI_KLASORU / "veri_kalite_ozeti.csv",
    encoding="utf-8-sig",
    index_label="sutun"
)

sayisal_kontrol.to_csv(
    CIKTI_KLASORU / "sayisal_deger_kontrolu.csv",
    encoding="utf-8-sig",
    index_label="sutun"
)

SÜTUNLAR VE VERİ TİPLERİ
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 1

,veri_tipi,eksik_deger,benzersiz_deger
Administrative,int64,0,27
Administrative_Duration,float64,0,3335
Informational,int64,0,17
Informational_Duration,float64,0,1258
ProductRelated,int64,0,311
ProductRelated_Duration,float64,0,9551
BounceRates,float64,0,1872
ExitRates,float64,0,4777
PageValues,float64,0,2704
SpecialDay,float64,0,6


,minimum,maksimum,negatif_deger,sonsuz_deger
Administrative,0.0,27.000000,0,0
Administrative_Duration,0.0,3398.750000,0,0
Informational,0.0,24.000000,0,0
Informational_Duration,0.0,2549.375000,0,0
ProductRelated,0.0,705.000000,0,0
ProductRelated_Duration,0.0,63973.522230,0,0
BounceRates,0.0,0.200000,0,0
ExitRates,0.0,0.200000,0,0
PageValues,0.0,361.763742,0,0
SpecialDay,0.0,1.000000,0,0


Toplam eksik hücre: 0
Tam satır tekrar sayısı: 125
Aynı girdilere rağmen farklı hedef içeren grup: 0


In [9]:
# Ham veri korunur. Tekilleştirilmiş veri ayrı bir değişkende tutulur.
df_temiz = df.drop_duplicates().copy()

temizleme_ozeti = pd.DataFrame({
    "islem": [
        "Başlangıç satır sayısı",
        "Çıkarılan tam tekrar sayısı",
        "Kalan satır sayısı"
    ],
    "deger": [
        len(df),
        len(df) - len(df_temiz),
        len(df_temiz)
    ]
})

display(temizleme_ozeti)

# Sorun varsa sessizce doldurmak veya silmek yerine duruyoruz.
if df_temiz.isna().any().any():
    raise ValueError(
        "Eksik değer bulundu. Model aşamasından önce uygun işlemi seçmeliyiz."
    )

sayisal_temiz = df_temiz.select_dtypes(include="number")

if not np.isfinite(sayisal_temiz.to_numpy()).all():
    raise ValueError("Sonsuz veya geçersiz sayısal değer bulundu.")

if (sayisal_temiz < 0).any().any():
    raise ValueError(
        "Negatif değer bulundu. İlgili sütun ve kayıtları incelemeliyiz."
    )

# Analiz kopyasını Drive'a kaydet.
df_temiz.to_csv(
    CIKTI_KLASORU / "oturumlar_tekillestirilmis.csv",
    index=False,
    encoding="utf-8-sig"
)

temizleme_ozeti.to_csv(
    CIKTI_KLASORU / "temizleme_ozeti.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Kontroller tamamlandı; tekilleştirilmiş veri kaydedildi.")

,islem,deger
0,Başlangıç satır sayısı,12330
1,Çıkarılan tam tekrar sayısı,125
2,Kalan satır sayısı,12205


Kontroller tamamlandı; tekilleştirilmiş veri kaydedildi.


In [10]:
from sklearn.model_selection import train_test_split

# Ana modelde PageValues kullanılmayacak.
ozellikler = [
    sutun for sutun in df_temiz.columns
    if sutun not in ["Revenue", "PageValues"]
]

X = df_temiz[ozellikler].copy()
y = df_temiz["Revenue"].astype(int).copy()

# PageValues çıkarıldıktan sonra da aynı girdiler oluşabilir.
# Bu durumda sıradan rastgele bölme yerine grup bazlı ayrım gerekir.
ayni_girdi_sayisi = int(X.duplicated().sum())
print("Seçilen model girdilerindeki tekrar sayısı:", ayni_girdi_sayisi)

if ayni_girdi_sayisi > 0:
    print(
        "Grup bazlı ayrım gerekiyor. "
        "Bu çıktıyı paylaş; sonraki hücreyi buna göre hazırlayacağız."
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    bolunme_ozeti = pd.DataFrame({
        "veri": ["Eğitim", "Test"],
        "satir_sayisi": [len(X_train), len(X_test)],
        "ozellik_sayisi": [X_train.shape[1], X_test.shape[1]],
        "satin_alma_orani_yuzde": [
            round(y_train.mean() * 100, 2),
            round(y_test.mean() * 100, 2)
        ]
    })

    display(bolunme_ozeti)

    # Analizleri bundan sonra eğitim tarafında yapacağız.
    egitim_df = X_train.copy()
    egitim_df["Revenue"] = y_train

    # Bölünmeyi yeniden takip edebilmek için satır indeksleri kaydedilir.
    bolunme_kaydi = pd.concat([
        pd.DataFrame({
            "kaynak_satir_indeksi": X_train.index,
            "bolum": "egitim"
        }),
        pd.DataFrame({
            "kaynak_satir_indeksi": X_test.index,
            "bolum": "test"
        })
    ], ignore_index=True)

    bolunme_kaydi.to_csv(
        CIKTI_KLASORU / "egitim_test_indeksleri.csv",
        index=False
    )

    bolunme_ozeti.to_csv(
        CIKTI_KLASORU / "egitim_test_ozeti.csv",
        index=False,
        encoding="utf-8-sig"
    )

Seçilen model girdilerindeki tekrar sayısı: 0


,veri,satir_sayisi,ozellik_sayisi,satin_alma_orani_yuzde
0,Eğitim,9764,16,15.63
1,Test,2441,16,15.65
